# Sequence-Based Data for Deep Learning: Sentiment Analysis with RNN

**Task:** Binary sentiment classification (positive / negative) on the IMDB movie review dataset.  
**Architecture:** Recurrent Neural Network (RNN) — the simplest sequence model, as introduced in topic 10.  
**Framework:** PyTorch

---

## Why sentiment analysis?

Text is a natural sequence: words depend on the words that came before them.  
A plain MLP cannot capture this — it sees each word independently.  
An RNN maintains a **hidden state** that accumulates context across the sequence, making it suitable for this task.

## Pipeline overview

```
Raw text → Tokenisation → Vocabulary → Embedding → RNN → Classifier
```

In [ ]:
import random
import re
import time
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import matplotlib.pyplot as plt

# reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 1. Imports and reproducibility

In [ ]:
try:
    from torchtext.datasets import IMDB
    from torchtext.data.utils import get_tokenizer
    print("torchtext available.")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torchtext", "--quiet"])
    from torchtext.datasets import IMDB
    from torchtext.data.utils import get_tokenizer
    print("torchtext installed and imported.")

In [ ]:
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

try:
    import datasets as hf_datasets
    print("datasets already installed.")
except ImportError:
    print("Installing Hugging Face datasets…")
    pip_install("datasets")
    import datasets as hf_datasets
    print("Done.")

In [ ]:
from datasets import load_dataset

print("Loading IMDB dataset (first run downloads ~80 MB) …")
imdb = load_dataset("imdb")   # splits: 'train', 'test'

# Each sample: {'text': str, 'label': int}  — 0=negative, 1=positive
print(f"Train samples : {len(imdb['train']):,}")
print(f"Test  samples : {len(imdb['test']):,}")
print("\nExample:")
print("Label :", imdb['train'][0]['label'], "(1 = positive)")
print("Text  :", imdb['train'][0]['text'][:200], "…")

Output:
Loading IMDB dataset (first run downloads ~80 MB) …
Train samples : 25,000
Test  samples : 25,000

Example:
Label : 0 (1 = positive)
Text  : I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev 

## 3. Tokenisation and vocabulary

We use a basic whitespace tokeniser and build a vocabulary from the training set.  
Words that appear fewer than `MIN_FREQ` times are replaced by `<unk>`.  
All sequences are padded to the same length with `<pad>`.

In [ ]:
# hyper-parameters
MAX_LEN = 256 # truncate reviews to this many tokens
MIN_FREQ = 5 # minimum token frequency to enter vocabulary
BATCH_SIZE = 64

tokenizer = get_tokenizer("basic_english")


def clean(text: str):
    """Remove HTML tags and lowercase."""
    text = re.sub(r"<[^>]+>", " ", text)
    return text.lower().strip()


# load raw data
print("Loading IMDB data (first run downloads ~8 MB) …")
train_iter = list(IMDB(split="train"))   # list of (label_str, text) tuples
test_iter = list(IMDB(split="test"))

# label: 'pos' → 1, 'neg' → 0
label_map = {"pos": 1, "neg": 0}

train_data = [(label_map[lbl], tokenizer(clean(txt))[:MAX_LEN])
              for lbl, txt in train_iter]
test_data = [(label_map[lbl], tokenizer(clean(txt))[:MAX_LEN])
              for lbl, txt in test_iter]

print(f"Train samples : {len(train_data):,}")
print(f"Test  samples : {len(test_data):,}")

In [ ]:
# build vocabulary from training data only
counter = Counter(tok for _, tokens in train_data for tok in tokens)

# special tokens
PAD_IDX, UNK_IDX = 0, 1
vocab = {"<pad>": PAD_IDX, "<unk>": UNK_IDX}
for word, freq in counter.items():
    if freq >= MIN_FREQ:
        vocab[word] = len(vocab)

VOCAB_SIZE = len(vocab)
print(f"Vocabulary size: {VOCAB_SIZE:,} tokens")


def encode(tokens):
    return [vocab.get(t, UNK_IDX) for t in tokens]

## 4. Dataset and DataLoader

We wrap our encoded data in a `torch.utils.data.Dataset` and use a custom `collate_fn` to pad sequences inside each batch to the same length.

This is the standard approach in PyTorch for variable-length sequence data.

In [ ]:
class IMDBDataset(Dataset):
    def __init__(self, data):
        self.samples = [(label, torch.tensor(encode(tokens), dtype=torch.long))
                        for label, tokens in data]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def collate_fn(batch):
    """Pad sequences in a batch to the same length."""
    labels, seqs = zip(*batch)
    padded = pad_sequence(seqs, batch_first=True, padding_value=PAD_IDX)
    return torch.tensor(labels, dtype=torch.float), padded


train_dataset = IMDBDataset(train_data)
test_dataset  = IMDBDataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)} | Test batches: {len(test_loader)}")

## 5. Model: Embedding + RNN + Classifier

The architecture follows directly from the slides:

```
Token IDs  →  Embedding layer  →  RNN (many-to-one)  →  Linear + Sigmoid
```

Many-to-one because we feed the full review and produce a single sentiment score at the end.

The hidden state at the last time step summarises the entire sequence, this is what we classify.

In [ ]:
class SentimentRNN(nn.Module):
    """
    Embedding → RNN (many-to-one) → Linear → Sigmoid

    Parameters
    ----------
    vocab_size   : number of unique tokens
    embed_dim    : size of each token embedding vector
    hidden_size  : number of features in the RNN hidden state
    num_layers   : stacked (deep) RNN depth
    dropout      : dropout probability (applied between RNN layers)
    pad_idx      : index of the <pad> token (ignored by embedding)
    """

    def __init__(self, vocab_size, embed_dim, hidden_size,
                 num_layers=1, dropout=0.3, pad_idx=PAD_IDX):
        super().__init__()

        # 1. Embedding: converts token IDs → dense vectors
        self.embedding = nn.Embedding(vocab_size, embed_dim,
                                      padding_idx=pad_idx)

        # 2. RNN: processes the sequence step-by-step
        # nonlinearity='tanh' is the classic choice (matches the slides)
        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,       # input shape: (batch, seq_len, embed_dim)
            dropout=dropout if num_layers > 1 else 0.0,
            nonlinearity="tanh",
        )

        # 3. Output head: hidden state → scalar sentiment score
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_size, 1)
        self.sigmoid   = nn.Sigmoid()

    def forward(self, x):
        # x: (batch, seq_len)
        embedded = self.dropout(self.embedding(x)) # (batch, seq_len, embed_dim)

        # output: (batch, seq_len, hidden)
        # hidden: (num_layers, batch, hidden)
        _, hidden = self.rnn(embedded)

        # Take the last layer's hidden state → (batch, hidden)
        last_hidden = self.dropout(hidden[-1])

        # Classify
        out = self.fc(last_hidden).squeeze(1) # (batch,)
        return self.sigmoid(out)

In [ ]:
# model hyper-parameters
EMBED_DIM   = 128
HIDDEN_SIZE = 256
NUM_LAYERS  = 2
DROPOUT     = 0.3
EPOCHS      = 5
LR          = 1e-3

model = SentimentRNN(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)

print(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

## 6. Loss function and optimiser

Binary cross-entropy (BCELoss) is the natural choice for two-class classification with a sigmoid output.  
Adam is used as the optimiser, it is the standard for RNN training.

In [ ]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

## 7. Training and evaluation helpers

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for labels, seqs in loader:
        labels = labels.to(device)
        seqs = seqs.to(device)

        optimizer.zero_grad()
        preds = model(seqs) # forward pass
        loss = criterion(preds, labels) # compute loss
        loss.backward() # BPTT (backprop through time)

        # Gradient clipping — prevents exploding gradients (key RNN problem)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += ((preds >= 0.5) == labels.bool()).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for labels, seqs in loader:
        labels = labels.to(device)
        seqs = seqs.to(device)

        preds = model(seqs)
        loss = criterion(preds, labels)

        total_loss += loss.item() * labels.size(0)
        correct += ((preds >= 0.5) == labels.bool()).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

## 8. Training loop

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

best_val_acc = 0.0
best_model_state = None

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc = evaluate(model, test_loader,  criterion, DEVICE)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    elapsed = time.time() - t0

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train loss {train_loss:.4f}  acc {train_acc:.3f} | "
        f"Val loss {val_loss:.4f}  acc {val_acc:.3f} | "
        f"{elapsed:.1f}s"
    )

    # save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

print(f"\nBest validation accuracy: {best_val_acc:.3f}")

## 9. Results visualisation

In [ ]:
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(epochs_range, history["train_loss"], label="Train", marker="o")
axes[0].plot(epochs_range, history["val_loss"],   label="Validation", marker="o")
axes[0].set_title("Loss per epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BCE Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, history["train_acc"], label="Train", marker="o")
axes[1].plot(epochs_range, history["val_acc"],   label="Validation", marker="o")
axes[1].set_title("Accuracy per epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()
print("Plot saved to training_curves.png")

## 10. Load best model and final test accuracy

In [ ]:
# Restore best checkpoint
model.load_state_dict(best_model_state)
model.to(DEVICE)

test_loss, test_acc = evaluate(model, test_loader, criterion, DEVICE)
print(f"Final test loss : {test_loss:.4f}")
print(f"Final test acc  : {test_acc:.3f}  ({test_acc*100:.1f}%)")

## 11. Inference on custom text

Let's test the model on a few reviews written from scratch.

In [ ]:
@torch.no_grad()
def predict(text: str, model=model, device=DEVICE) -> dict:
    model.eval()
    tokens  = tokenizer(clean(text))[:MAX_LEN]
    ids     = torch.tensor(encode(tokens), dtype=torch.long).unsqueeze(0).to(device)
    prob    = model(ids).item()
    label   = "POSITIVE" if prob >= 0.5 else "NEGATIVE"
    return {"label": label, "confidence": max(prob, 1 - prob)}


reviews = [
    "This movie was absolutely wonderful! The acting was superb and the story kept me hooked.",
    "Terrible film. The plot made no sense and the characters were completely flat.",
    "It was okay I guess, nothing special but not terrible either.",
    "One of the best films I have ever seen. A masterpiece of modern cinema.",
]

for rev in reviews:
    result = predict(rev)
    print(f"[{result['label']}  {result['confidence']:.2f}]  {rev[:70]}…")

## 12. Save the model

In [ ]:
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "vocab": vocab,
        "hyperparams": {
            "vocab_size":   VOCAB_SIZE,
            "embed_dim":    EMBED_DIM,
            "hidden_size":  HIDDEN_SIZE,
            "num_layers":   NUM_LAYERS,
            "dropout":      DROPOUT,
        },
    },
    "model.pt",
)
print("Model saved to model.pt")